# Post-Hoc FDP Bounds

This notebook shows the smallest workflow for attaching a post-hoc FDP certificate to unweighted empirical split conformal p-values.

## Import

This section loads the dependencies used throughout the notebook.

In [1]:
import logging

import numpy as np
import pandas as pd
from oddball import Dataset, load
from pyod.models.iforest import IForest

from nonconform import ConformalDetector, Split
from nonconform.fdr import conformal_fdp_upper_bound_from_result
from nonconform.metrics import false_discovery_rate, statistical_power

root_logger = logging.getLogger("nonconform")
if not root_logger.handlers:
    root_logger.addHandler(logging.NullHandler())
root_logger.setLevel(logging.ERROR)

## Setup

We load Shuttle with a fixed seed and choose a few p-value thresholds to certify after p-values are computed.

In [2]:
x_train, x_test, y_test = load(Dataset.SHUTTLE, setup=True, seed=42)

n_calib = 1_000
thresholds = np.array([0.005, 0.01, 0.02, 0.05, 0.1])

print(f"x_train: {x_train.shape}, x_test: {x_test.shape}")
print(f"y_test positives: {int(y_test.sum())}")
print(f"calibration size={n_calib}")

x_train: (22793, 9), x_test: (1000, 9)
y_test positives: 100
calibration size=1000


## FDP Certificate

`conformal_fdp_upper_bound_from_result(...)` uses the cached `last_result` from `compute_p_values(...)` and returns threshold-level FDP and precision certificates.

In [3]:
detector = ConformalDetector(
    detector=IForest(n_estimators=100, max_samples=0.8, random_state=42),
    strategy=Split(n_calib=n_calib),
    seed=42,
)

detector.fit(x_train)
p_values = detector.compute_p_values(x_test)
bounds = conformal_fdp_upper_bound_from_result(
    detector.last_result,
    confidence=0.95,
    method="mc_thc",
    seed=42,
    thresholds=thresholds,
)

print(f"method={bounds.method}, confidence={bounds.confidence}")
print(bounds.to_frame().to_string(index=False, float_format=lambda x: f"{x:.3f}"))

method=mc_thc, confidence=0.95
 threshold  discoveries  fdp_upper_bound  precision_lower_bound
     0.005          103            0.226                  0.774
     0.010          111            0.210                  0.790
     0.020          120            0.259                  0.741
     0.050          154            0.423                  0.577
     0.100          212            0.581                  0.419


## Use a Threshold

`select(...)` applies the threshold. `bound_at(...)` and `precision_at(...)` attach the post-hoc certificate to that same threshold. The empirical columns below use labels only to check this benchmark example.

In [4]:
threshold = 0.05
selected = bounds.select(threshold)

summary = pd.DataFrame(
    [
        {
            "threshold": threshold,
            "discoveries": int(selected.sum()),
            "fdp_upper_bound": bounds.bound_at(threshold),
            "precision_lower_bound": bounds.precision_at(threshold),
            "empirical_fdr": false_discovery_rate(y_test, selected),
            "power": statistical_power(y_test, selected),
        }
    ]
)

print(summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

 threshold  discoveries  fdp_upper_bound  precision_lower_bound  empirical_fdr  power
     0.050          154            0.423                  0.577          0.351  1.000


## Interpretation

Read `fdp_upper_bound` as a high-confidence cap on the false-positive fraction among the selected points at that threshold. `precision_lower_bound` is the matching conservative minimum precision. Use the table to choose a practical trade-off, then report the threshold and certificate together. This is different from `detector.select(..., alpha=...)`, which is the fixed-level FDR-control workflow.